# Dimensional Modeling and Star Schemas

## Quick notes
### What this covers
- How to shape data for reporting (OLAP) instead of for day-to-day writes (OLTP): facts, dimensions, grain, star vs snowflake.
- Building a small sales star from an OLTP schema with plain `INSERT ... SELECT` (ETL in SQL), including a date dimension made by a recursive CTE.
- Fact table types, additive vs non-additive measures, slowly changing dimensions (types 1, 2, 3), junk and degenerate dimensions, bridge tables, late-arriving data, conformed dimensions.
- The BI queries a star is built for, and when a star is overkill.
- Normal forms, keys and relationship modelling are NOT repeated here: see 09-Design/01-Normalization-And-Design. This notebook is the "other direction": on purpose, we denormalize for reading.
### Key points
- OLTP = many small writes, normalized, protects correctness. OLAP = big reads and aggregates, denormalized, protects speed and simplicity of queries.
- A **fact** table holds numbers about events (quantity, amount). **Dimension** tables hold the descriptive context (who, what, when, where).
- **Declare the grain first**: one sentence saying what one fact row means ("one row per order line"). Everything else follows from it.
- A star has one fact in the middle and flat, wide dimensions around it, one join deep. A snowflake normalizes the dimensions into more tables.
- Use **surrogate keys** (meaningless integers) in dimensions, keep the business key as a plain column.
- Slowly changing dimensions: type 1 overwrite (no history), type 2 add a new row (full history), type 3 add a "previous value" column (one step of history).
### Gotchas (the part worth revising twice)
- Mixing grains in one fact table (an order-level fee on line-level rows) double counts when you SUM it.
- Adding up a balance (stock on hand) across dates is wrong. It is semi-additive: sum across products, but average or take the last across time.
- Averaging ratios (margin %) is wrong. Add the parts, then divide: `SUM(margin) / SUM(revenue)`.
- A bridge table for many-to-many fans out your facts. Use a weighting factor or you overcount.
- Joining two fact tables directly multiplies rows. Aggregate each fact to the shared dimension first (drill across).
- A NULL foreign key in a fact row silently drops it from inner joins. Point it at an "Unknown" dimension row instead.
- SCD type 2 means the fact must pick the dimension row that was valid on the event date, not just the current one.
### When it matters
- Dashboards, monthly reports, "revenue by category by month", anything that scans a lot of history and groups it many ways.
- Any time analysts keep writing the same 6-table join to answer a simple question.
### One-line recap example
- `SELECT d.year_month, p.category, SUM(f.gross_amount) FROM fact_sales f JOIN dim_date d USING (date_key) JOIN dim_product p USING (product_key) GROUP BY 1, 2;` means "numbers from the fact, labels from the dimensions, one join each".

## 01. OLTP vs OLAP, and the seed OLTP schema
- OLTP (online transaction processing) is the app database: place an order, change an address. Tables are normalized so each fact lives in one place.
- OLAP (online analytical processing) is reporting: "revenue by category by month". Queries read millions of rows and group them.
- The same data is usually kept twice: the OLTP tables for the app, and a star schema (built from them) for reporting.
- Below is a small shop OLTP schema. It is our source system for the rest of the notebook (tables prefixed `oltp_`).

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE oltp_customers (
    customer_id INTEGER PRIMARY KEY, name TEXT NOT NULL, city TEXT NOT NULL, segment TEXT NOT NULL
);
CREATE TABLE oltp_products (
    product_id INTEGER PRIMARY KEY, sku TEXT NOT NULL UNIQUE, name TEXT NOT NULL,
    category TEXT NOT NULL, unit_cost INTEGER NOT NULL
);
CREATE TABLE oltp_orders (
    order_id INTEGER PRIMARY KEY, customer_id INTEGER NOT NULL REFERENCES oltp_customers,
    order_date TEXT NOT NULL, ship_date TEXT, delivered_date TEXT,
    channel TEXT NOT NULL, payment_method TEXT NOT NULL, shipping_fee INTEGER NOT NULL
);
CREATE TABLE oltp_order_lines (
    line_id INTEGER PRIMARY KEY, order_id INTEGER NOT NULL REFERENCES oltp_orders,
    product_id INTEGER NOT NULL REFERENCES oltp_products, qty INTEGER NOT NULL, unit_price INTEGER NOT NULL
);
INSERT INTO oltp_customers VALUES
    (1,'Asha','Pune','Retail'),(2,'Ben','Delhi','Retail'),(3,'Chen','Mumbai','Corporate'),(4,'Dina','Pune','Corporate');
INSERT INTO oltp_products VALUES
    (1,'P-100','Notebook','Stationery',30),(2,'P-200','Pen','Stationery',4),
    (3,'P-300','Backpack','Bags',500),(4,'P-400','Desk Lamp','Home',350),(5,'P-500','Mug','Home',80);
INSERT INTO oltp_orders VALUES
    (1001,1,'2025-01-05','2025-01-06','2025-01-09','web','card',40),
    (1002,2,'2025-01-18','2025-01-19','2025-01-24','store','cash',0),
    (1003,3,'2025-01-29','2025-01-30','2025-02-02','web','card',50),
    (1004,1,'2025-02-10','2025-02-11','2025-02-13','app','upi',40),
    (1005,4,'2025-02-21','2025-02-25',NULL,'web','upi',60),
    (1006,2,'2025-03-04',NULL,NULL,'web','card',40),
    (1007,3,'2025-03-12','2025-03-13','2025-03-17','store','cash',0),
    (1008,1,'2025-03-25','2025-03-26','2025-03-28','app','card',40);
INSERT INTO oltp_order_lines (order_id, product_id, qty, unit_price) VALUES
    (1001,1,4,50),(1001,2,10,10),(1002,3,1,800),(1003,4,1,600),(1003,5,2,150),
    (1004,1,2,50),(1004,5,4,150),(1005,3,2,780),(1006,2,20,9),(1006,1,5,50),
    (1007,4,2,600),(1007,3,1,800),(1008,5,3,150),(1008,2,5,10);
SELECT count(*) AS orders, (SELECT count(*) FROM oltp_order_lines) AS lines FROM oltp_orders;

orders | lines
-------+------
8      | 14   
(1 row)



## 02. The same question on OLTP: it works, but it is clumsy
- Question: "revenue and margin by month and product category". On the normalized schema that needs a join through lines, orders and products, and date parsing in every query.
- Fine for one report. But every analyst re-writes this join (and every one can get it wrong), and it scans live app tables.
- Remember this query. We will answer the same question from a star later, and it will be much shorter.

In [3]:
%%sql
SELECT substr(o.order_date, 1, 7) AS year_month, p.category,
       SUM(l.qty * l.unit_price) AS revenue,
       SUM(l.qty * (l.unit_price - p.unit_cost)) AS margin
FROM oltp_order_lines l
JOIN oltp_orders   o ON o.order_id   = l.order_id
JOIN oltp_products p ON p.product_id = l.product_id
GROUP BY 1, 2
ORDER BY 1, 2;

year_month | category   | revenue | margin
-----------+------------+---------+-------
2025-01    | Bags       | 800     | 300   
2025-01    | Home       | 900     | 390   
2025-01    | Stationery | 300     | 140   
2025-02    | Bags       | 1560    | 560   
2025-02    | Home       | 600     | 280   
2025-02    | Stationery | 100     | 40    
2025-03    | Bags       | 800     | 300   
2025-03    | Home       | 1650    | 710   
2025-03    | Stationery | 480     | 230   
(9 rows)



## 03. Facts and dimensions
- A **fact** row records something that happened and carries numbers: quantity sold, amount paid. Facts are narrow and numerous (millions of rows).
- A **dimension** row describes a thing you filter or group by: a customer, a product, a day. Dimensions are wide (many descriptive columns) and small (thousands of rows).
- A fact row holds a foreign key to each dimension plus the measures. Pictured, that is a star:
```
                dim_date
                   |
  dim_customer -- fact_sales -- dim_product
                   |
             dim_order_profile
```
- Rule of thumb: if you would `SUM` it, it is a measure (fact). If you would `GROUP BY` or `WHERE` on it, it is an attribute (dimension).

## 04. Declare the grain first
- **Grain** = what one row of the fact table means, in one plain sentence. Our grain: "one row per order line".
- Declare it before listing columns. Every measure and every dimension key must be true at that grain.
- Bad: a measure that belongs to a coarser grain. `shipping_fee` is per ORDER, but an order with 2 lines has 2 rows. Summing it counts the fee twice.
- Below: the true total shipping is 270. Putting the fee on the line rows gives a bigger, wrong number.

In [4]:
%%sql
SELECT (SELECT SUM(shipping_fee) FROM oltp_orders)            AS true_shipping,
       SUM(o.shipping_fee)                                      AS summed_on_line_rows
FROM oltp_order_lines l JOIN oltp_orders o ON o.order_id = l.order_id;

true_shipping | summed_on_line_rows
--------------+--------------------
270           | 480                
(1 row)



- Fix: do not put order-level measures on line-level rows. Give them their own fact at the order grain (we build one later: the accumulating snapshot, section 14), or allocate the fee across lines on purpose and document it.

## 05. Star vs snowflake
- **Star**: each dimension is ONE flat table. `dim_product` repeats the category name on every product row. Queries are one join deep.
- **Snowflake**: the dimension is normalized into sub-tables (`dim_product -> dim_category -> dim_department`). Less repetition, but more joins.
- Default to a star. Storage is cheap, joins and analyst confusion are not. Snowflake only when a sub-dimension is shared widely or is huge.
- Below is the snowflake shape of the product dimension: one extra join just to reach the category name. The star shape comes next.

In [5]:
%%sql
CREATE TABLE snow_category (category_key INTEGER PRIMARY KEY, category TEXT NOT NULL UNIQUE);
CREATE TABLE snow_product (
    product_key INTEGER PRIMARY KEY, sku TEXT NOT NULL, product_name TEXT NOT NULL,
    category_key INTEGER NOT NULL REFERENCES snow_category
);
INSERT INTO snow_category (category) SELECT DISTINCT category FROM oltp_products ORDER BY category;
INSERT INTO snow_product (sku, product_name, category_key)
SELECT p.sku, p.name, c.category_key FROM oltp_products p JOIN snow_category c USING (category) ORDER BY p.sku;
-- snowflake: one extra join to reach the category name
SELECT c.category, count(*) AS products
FROM snow_product sp JOIN snow_category c USING (category_key)
GROUP BY c.category ORDER BY c.category;

category   | products
-----------+---------
Bags       | 1       
Home       | 2       
Stationery | 2       
(3 rows)



- In the star we skip `snow_category` and keep `category` as a column of `dim_product`. That is the shape we build next. (Drop the snowflake demo, we do not need it again.)

In [6]:
%%sql
DROP TABLE snow_product;
DROP TABLE snow_category;

## 06. Surrogate keys and the first dimensions
- A **surrogate key** is a meaningless integer made by the warehouse (`customer_key`). The business key from the source (`customer_id`) stays as a normal column.
- Why: source keys can be reused, change format, or collide across systems; and with SCD type 2 one customer needs several rows, so `customer_id` alone cannot be the key.
- Add an **Unknown member** per dimension (key `-1`) so a fact never needs a NULL foreign key.
- `dim_customer` is built with the SCD-2 columns (`valid_from`, `valid_to`, `is_current`) from day one. Section 17 explains them; for now every customer has one row. Rows are valid in the half-open range `[valid_from, valid_to)`.

In [7]:
%%sql
CREATE TABLE dim_customer (
    customer_key  INTEGER PRIMARY KEY,
    customer_id   INTEGER,              -- business key from the source (NULL for the unknown member)
    customer_name TEXT NOT NULL,
    city          TEXT NOT NULL,
    segment       TEXT NOT NULL,
    valid_from    TEXT NOT NULL,
    valid_to      TEXT NOT NULL,        -- '9999-12-31' = still current
    is_current    INTEGER NOT NULL CHECK (is_current IN (0, 1)),
    UNIQUE (customer_id, valid_from)
);
-- at most one current row per customer
CREATE UNIQUE INDEX ux_dim_customer_current ON dim_customer (customer_id) WHERE is_current = 1;

CREATE TABLE dim_product (
    product_key  INTEGER PRIMARY KEY,
    product_id   INTEGER UNIQUE,
    sku          TEXT NOT NULL,
    product_name TEXT NOT NULL,
    category     TEXT NOT NULL
);
-- load products (type-1 style: one row per product)
INSERT INTO dim_product (product_id, sku, product_name, category)
SELECT product_id, sku, name, category FROM oltp_products ORDER BY product_id;

-- load customers: one statement that works for the first load AND later changes (see section 17)
INSERT INTO dim_customer (customer_id, customer_name, city, segment, valid_from, valid_to, is_current)
SELECT c.customer_id, c.name, c.city, c.segment,
       CASE WHEN EXISTS (SELECT 1 FROM dim_customer d WHERE d.customer_id = c.customer_id)
            THEN '2025-01-01' ELSE '1900-01-01' END,
       '9999-12-31', 1
FROM oltp_customers c
WHERE NOT EXISTS (SELECT 1 FROM dim_customer d WHERE d.customer_id = c.customer_id AND d.is_current = 1)
ORDER BY c.customer_id;
-- the Unknown members go in after the real rows, so real keys start at 1
INSERT INTO dim_customer VALUES (-1, NULL, 'Unknown', 'Unknown', 'Unknown', '1900-01-01', '9999-12-31', 1);
INSERT INTO dim_product  VALUES (-1, NULL, 'n/a', 'Unknown', 'Unknown');
SELECT customer_key, customer_id, customer_name, city, valid_from, valid_to, is_current FROM dim_customer ORDER BY customer_key;

customer_key | customer_id | customer_name | city    | valid_from | valid_to   | is_current
-------------+-------------+---------------+---------+------------+------------+-----------
-1           | NULL        | Unknown       | Unknown | 1900-01-01 | 9999-12-31 | 1         
1            | 1           | Asha          | Pune    | 1900-01-01 | 9999-12-31 | 1         
2            | 2           | Ben           | Delhi   | 1900-01-01 | 9999-12-31 | 1         
3            | 3           | Chen          | Mumbai  | 1900-01-01 | 9999-12-31 | 1         
4            | 4           | Dina          | Pune    | 1900-01-01 | 9999-12-31 | 1         
(5 rows)



## 07. A date dimension from a recursive CTE
- Almost every fact has a date, and analysts want "by month", "by quarter", "weekends only". Precompute those labels once in `dim_date`, one row per day.
- SQLite has no `generate_series`, so a recursive CTE walks day by day (see 02-Querying/03-Subqueries-And-CTEs).
- Key convention: `date_key` is an integer like `20250131`. It sorts correctly, reads well, and needs no join to filter a range.
- Add a row `date_key = 0` for "not yet happened" (used later for a shipment that has not shipped).

In [8]:
%%sql
CREATE TABLE dim_date (
    date_key    INTEGER PRIMARY KEY,   -- 20250131
    full_date   TEXT UNIQUE,           -- '2025-01-31'
    year        INTEGER, quarter INTEGER, month INTEGER, day INTEGER,
    year_month  TEXT,                  -- '2025-01' (sorts, and is handy for GROUP BY)
    month_name  TEXT, day_of_week INTEGER, day_name TEXT, is_weekend INTEGER
);
INSERT INTO dim_date
WITH RECURSIVE d(dt) AS (
    SELECT '2025-01-01' UNION ALL SELECT date(dt, '+1 day') FROM d WHERE dt < '2025-03-31'
)
SELECT CAST(strftime('%Y%m%d', dt) AS INTEGER), dt,
       CAST(strftime('%Y', dt) AS INTEGER),
       (CAST(strftime('%m', dt) AS INTEGER) + 2) / 3,
       CAST(strftime('%m', dt) AS INTEGER), CAST(strftime('%d', dt) AS INTEGER),
       strftime('%Y-%m', dt),
       substr('JanFebMarAprMayJunJulAugSepOctNovDec', (CAST(strftime('%m', dt) AS INTEGER) - 1) * 3 + 1, 3),
       CAST(strftime('%w', dt) AS INTEGER),                       -- 0 = Sunday
       substr('SunMonTueWedThuFriSat', CAST(strftime('%w', dt) AS INTEGER) * 3 + 1, 3),
       CASE WHEN strftime('%w', dt) IN ('0', '6') THEN 1 ELSE 0 END
FROM d;
INSERT INTO dim_date (date_key, full_date, year_month, month_name, day_name) VALUES (0, NULL, 'n/a', 'n/a', 'n/a');
SELECT count(*) AS days, min(full_date) AS first_day, max(full_date) AS last_day FROM dim_date WHERE date_key > 0;
SELECT * FROM dim_date WHERE date_key IN (0, 20250101, 20250215, 20250331) ORDER BY date_key;

days | first_day  | last_day  
-----+------------+-----------
90   | 2025-01-01 | 2025-03-31
(1 row)

date_key | full_date  | year | quarter | month | day  | year_month | month_name | day_of_week | day_name | is_weekend
---------+------------+------+---------+-------+------+------------+------------+-------------+----------+-----------
0        | NULL       | NULL | NULL    | NULL  | NULL | n/a        | n/a        | NULL        | n/a      | NULL      
20250101 | 2025-01-01 | 2025 | 1       | 1     | 1    | 2025-01    | Jan        | 3           | Wed      | 0         
20250215 | 2025-02-15 | 2025 | 1       | 2     | 15   | 2025-02    | Feb        | 6           | Sat      | 1         
20250331 | 2025-03-31 | 2025 | 1       | 3     | 31   | 2025-03    | Mar        | 1           | Mon      | 0         
(4 rows)



- For real use, extend the end date to cover several years ahead, add fiscal periods and holiday flags. The one-off cost is tiny (365 rows a year).

## 08. A junk dimension for small flags
- Columns like `channel` (web/store/app) and `payment_method` (card/cash/upi) are low-cardinality labels. One tiny dimension per label is overkill, and putting the text on the fact makes it fat.
- A **junk dimension** holds every combination of such labels in one small table, and the fact stores one `profile_key`.
- Build it as the cross product of the distinct values (3 x 3 = 9 rows, plus the Unknown row). Combos that never occur are harmless.

In [9]:
%%sql
CREATE TABLE dim_order_profile (
    profile_key INTEGER PRIMARY KEY, channel TEXT NOT NULL, payment_method TEXT NOT NULL,
    UNIQUE (channel, payment_method)
);
INSERT INTO dim_order_profile (channel, payment_method)
SELECT c.channel, p.payment_method
FROM (SELECT DISTINCT channel FROM oltp_orders) c
CROSS JOIN (SELECT DISTINCT payment_method FROM oltp_orders) p
ORDER BY c.channel, p.payment_method;
INSERT INTO dim_order_profile VALUES (-1, 'Unknown', 'Unknown');
SELECT * FROM dim_order_profile ORDER BY profile_key;

profile_key | channel | payment_method
------------+---------+---------------
-1          | Unknown | Unknown       
1           | app     | card          
2           | app     | cash          
3           | app     | upi           
4           | store   | card          
5           | store   | cash          
6           | store   | upi           
7           | web     | card          
8           | web     | cash          
9           | web     | upi           
(10 rows)



## 09. The fact table and degenerate dimensions
- Grain: **one row per order line**. Keys: date, customer, product, order profile. Measures: `quantity`, `gross_amount`, `cost_amount`.
- `order_number` and `line_number` are a **degenerate dimension**: an identifier from the source that has no attributes of its own, so it needs no dimension table. It lives in the fact, and is handy for "show me all lines of order 1004" and as the primary key.
- Store amounts that you can sum (revenue, cost). Margin is derived (`gross_amount - cost_amount`), so we do not store a ratio.
- Foreign keys are on: a fact cannot point at a missing dimension row.

In [10]:
%%sql
CREATE TABLE fact_sales (
    order_number INTEGER NOT NULL,       -- degenerate dimension
    line_number  INTEGER NOT NULL,       -- degenerate dimension
    date_key     INTEGER NOT NULL REFERENCES dim_date,
    customer_key INTEGER NOT NULL REFERENCES dim_customer,
    product_key  INTEGER NOT NULL REFERENCES dim_product,
    profile_key  INTEGER NOT NULL REFERENCES dim_order_profile,
    quantity     INTEGER NOT NULL,
    gross_amount INTEGER NOT NULL,
    cost_amount  INTEGER NOT NULL,
    PRIMARY KEY (order_number, line_number)
) WITHOUT ROWID;
CREATE INDEX ix_fact_sales_date ON fact_sales (date_key);
CREATE INDEX ix_fact_sales_product ON fact_sales (product_key);
CREATE INDEX ix_fact_sales_customer ON fact_sales (customer_key);

## 10. ETL in SQL: load the fact with INSERT ... SELECT
- ETL = extract, transform, load. In SQLite the whole thing is one `INSERT ... SELECT` that joins the source to each dimension to swap business keys for surrogate keys.
- `LEFT JOIN` + `COALESCE(key, -1)` sends unknown or late dimension members to the Unknown row instead of dropping the fact.
- The customer lookup uses the date range, so the fact gets the customer version that was valid on the order date (matters once we have SCD type 2).
- `line_number` is computed with `ROW_NUMBER()` per order, because the source `line_id` is global.
- This statement is re-runnable: `DELETE` first, then insert (a full rebuild). Incremental loads filter on a watermark, shown in section 20.

In [11]:
%%sql
DELETE FROM fact_sales;
INSERT INTO fact_sales
SELECT o.order_id,
       ROW_NUMBER() OVER (PARTITION BY l.order_id ORDER BY l.line_id),
       dd.date_key,
       COALESCE(dc.customer_key, -1),
       COALESCE(dp.product_key, -1),
       COALESCE(pr.profile_key, -1),
       l.qty,
       l.qty * l.unit_price,
       l.qty * p.unit_cost
FROM oltp_order_lines l
JOIN oltp_orders   o  ON o.order_id = l.order_id
JOIN oltp_products p  ON p.product_id = l.product_id
JOIN dim_date      dd ON dd.full_date = o.order_date
LEFT JOIN dim_customer      dc ON dc.customer_id = o.customer_id
                              AND o.order_date >= dc.valid_from AND o.order_date < dc.valid_to
LEFT JOIN dim_product       dp ON dp.product_id = l.product_id
LEFT JOIN dim_order_profile pr ON pr.channel = o.channel AND pr.payment_method = o.payment_method;
SELECT count(*) AS fact_rows, SUM(gross_amount) AS revenue, SUM(cost_amount) AS cost FROM fact_sales;
SELECT * FROM fact_sales ORDER BY order_number, line_number LIMIT 4;

fact_rows | revenue | cost
----------+---------+-----
14        | 7190    | 4240
(1 row)

order_number | line_number | date_key | customer_key | product_key | profile_key | quantity | gross_amount | cost_amount
-------------+-------------+----------+--------------+-------------+-------------+----------+--------------+------------
1001         | 1           | 20250105 | 1            | 1           | 7           | 4        | 200          | 120        
1001         | 2           | 20250105 | 1            | 2           | 7           | 10       | 100          | 40         
1002         | 1           | 20250118 | 2            | 3           | 5           | 1        | 800          | 500        
1003         | 1           | 20250129 | 3            | 4           | 7           | 1        | 600          | 350        
(4 rows)



- Check the load: fact revenue must equal the source revenue. Always reconcile row counts and totals after an ETL step.

In [12]:
%%sql
SELECT (SELECT SUM(qty * unit_price) FROM oltp_order_lines) AS source_revenue,
       (SELECT SUM(gross_amount) FROM fact_sales)           AS fact_revenue,
       (SELECT count(*) FROM oltp_order_lines)              AS source_lines,
       (SELECT count(*) FROM fact_sales)                    AS fact_lines;

source_revenue | fact_revenue | source_lines | fact_lines
---------------+--------------+--------------+-----------
7190           | 7190         | 14           | 14        
(1 row)



## 11. The BI queries a star is built for
- Same question as section 02, now from the star: one join per dimension, readable labels, no date parsing.
- Note the pattern: `SELECT` dimension labels, `SUM` fact measures, `GROUP BY` the labels.

In [13]:
%%sql
SELECT d.year_month, p.category,
       SUM(f.gross_amount) AS revenue,
       SUM(f.gross_amount - f.cost_amount) AS margin
FROM fact_sales f
JOIN dim_date d    ON d.date_key = f.date_key
JOIN dim_product p ON p.product_key = f.product_key
GROUP BY d.year_month, p.category
ORDER BY d.year_month, p.category;

year_month | category   | revenue | margin
-----------+------------+---------+-------
2025-01    | Bags       | 800     | 300   
2025-01    | Home       | 900     | 390   
2025-01    | Stationery | 300     | 140   
2025-02    | Bags       | 1560    | 560   
2025-02    | Home       | 600     | 280   
2025-02    | Stationery | 100     | 40    
2025-03    | Bags       | 800     | 300   
2025-03    | Home       | 1650    | 710   
2025-03    | Stationery | 480     | 230   
(9 rows)



- **Rollup with subtotals**: SQLite has no `ROLLUP`, so add a subtotal level with `UNION ALL` (see 02-Querying/02-Aggregation-And-Grouping).

In [14]:
%%sql
SELECT d.year_month, p.category, SUM(f.gross_amount) AS revenue
FROM fact_sales f JOIN dim_date d USING (date_key) JOIN dim_product p USING (product_key)
GROUP BY d.year_month, p.category
UNION ALL
SELECT d.year_month, '(all categories)', SUM(f.gross_amount)
FROM fact_sales f JOIN dim_date d USING (date_key)
GROUP BY d.year_month
UNION ALL
SELECT '(all months)', '(all categories)', SUM(gross_amount) FROM fact_sales
ORDER BY 1, 2;

year_month   | category         | revenue
-------------+------------------+--------
(all months) | (all categories) | 7190   
2025-01      | (all categories) | 2000   
2025-01      | Bags             | 800    
2025-01      | Home             | 900    
2025-01      | Stationery       | 300    
2025-02      | (all categories) | 2260   
2025-02      | Bags             | 1560   
2025-02      | Home             | 600    
2025-02      | Stationery       | 100    
2025-03      | (all categories) | 2930   
2025-03      | Bags             | 800    
2025-03      | Home             | 1650   
2025-03      | Stationery       | 480    
(13 rows)



- **Ratio to total**: each category's share of all revenue, using a window over the grouped result (see 03-Window-Functions/01-Window-Functions).
- **Top-N per dimension**: the best-selling product inside each category. Rank inside the category, keep rank 1.

In [15]:
%%sql
SELECT p.category, SUM(f.gross_amount) AS revenue,
       ROUND(100.0 * SUM(f.gross_amount) / SUM(SUM(f.gross_amount)) OVER (), 1) AS pct_of_total
FROM fact_sales f JOIN dim_product p USING (product_key)
GROUP BY p.category ORDER BY revenue DESC;

WITH per_product AS (
    SELECT p.category, p.product_name, SUM(f.gross_amount) AS revenue
    FROM fact_sales f JOIN dim_product p USING (product_key)
    GROUP BY p.category, p.product_name
), ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY category ORDER BY revenue DESC, product_name) AS rn FROM per_product
)
SELECT category, product_name, revenue FROM ranked WHERE rn = 1 ORDER BY category;

category   | revenue | pct_of_total
-----------+---------+-------------
Bags       | 3160    | 43.9        
Home       | 3150    | 43.8        
Stationery | 880     | 12.2        
(3 rows)

category   | product_name | revenue
-----------+--------------+--------
Bags       | Backpack     | 3160   
Home       | Desk Lamp    | 1800   
Stationery | Notebook     | 550    
(3 rows)



## 12. Additive, semi-additive and non-additive measures
- **Additive**: can be summed across every dimension. `gross_amount`, `quantity`.
- **Semi-additive**: can be summed across SOME dimensions but not time. Account balance, stock on hand: summing Monday + Tuesday stock is nonsense. Across time use the last value, or an average.
- **Non-additive**: never summed. Ratios and percentages (margin %, unit price). Store the parts (margin, revenue) and divide after summing.
- Demo: a month-end stock snapshot fact (grain: one row per product per month-end). Its measure `qty_on_hand` is semi-additive.

In [16]:
%%sql
CREATE TABLE fact_stock_month_end (
    date_key INTEGER NOT NULL REFERENCES dim_date, product_key INTEGER NOT NULL REFERENCES dim_product,
    qty_on_hand INTEGER NOT NULL, PRIMARY KEY (date_key, product_key)
) WITHOUT ROWID;
INSERT INTO fact_stock_month_end
WITH s(sku, d, q) AS (VALUES
    ('P-100','2025-01-31',120),('P-100','2025-02-28',100),('P-100','2025-03-31',90),
    ('P-200','2025-01-31',400),('P-200','2025-02-28',350),('P-200','2025-03-31',300),
    ('P-300','2025-01-31',15),('P-300','2025-02-28',10),('P-300','2025-03-31',8),
    ('P-400','2025-01-31',20),('P-400','2025-02-28',20),('P-400','2025-03-31',12),
    ('P-500','2025-01-31',60),('P-500','2025-02-28',55),('P-500','2025-03-31',40))
SELECT dd.date_key, p.product_key, s.q FROM s
JOIN dim_date dd ON dd.full_date = s.d JOIN dim_product p ON p.sku = s.sku;
-- right: sum across products for ONE date
SELECT d.full_date, SUM(f.qty_on_hand) AS total_units_in_stock
FROM fact_stock_month_end f JOIN dim_date d USING (date_key)
GROUP BY d.full_date ORDER BY d.full_date;

full_date  | total_units_in_stock
-----------+---------------------
2025-01-31 | 615                 
2025-02-28 | 535                 
2025-03-31 | 450                 
(3 rows)



- Wrong vs right for one product over time. Summing 3 month-ends gives a number that means nothing (it is not "units in stock"). Use the latest value, or an average.

In [17]:
%%sql
SELECT p.product_name,
       SUM(f.qty_on_hand)  AS wrong_sum_over_time,
       ROUND(AVG(f.qty_on_hand), 1) AS avg_over_time,
       (SELECT f2.qty_on_hand FROM fact_stock_month_end f2
        WHERE f2.product_key = f.product_key ORDER BY f2.date_key DESC LIMIT 1) AS latest
FROM fact_stock_month_end f JOIN dim_product p USING (product_key)
GROUP BY f.product_key ORDER BY p.product_name;

product_name | wrong_sum_over_time | avg_over_time | latest
-------------+---------------------+---------------+-------
Backpack     | 33                  | 11.0          | 8     
Desk Lamp    | 52                  | 17.3          | 12    
Mug          | 155                 | 51.7          | 40    
Notebook     | 310                 | 103.3         | 90    
Pen          | 1050                | 350.0         | 300   
(5 rows)



- Non-additive: margin %. Averaging per-row percentages is wrong because big rows and small rows count the same. Sum the parts, then divide.

In [18]:
%%sql
SELECT p.category,
       ROUND(AVG(100.0 * (f.gross_amount - f.cost_amount) / f.gross_amount), 1) AS wrong_avg_of_pct,
       ROUND(100.0 * SUM(f.gross_amount - f.cost_amount) / SUM(f.gross_amount), 1) AS right_pct
FROM fact_sales f JOIN dim_product p USING (product_key)
GROUP BY p.category ORDER BY p.category;

category   | wrong_avg_of_pct | right_pct
-----------+------------------+----------
Bags       | 37.0             | 36.7     
Home       | 44.7             | 43.8     
Stationery | 49.3             | 46.6     
(3 rows)



## 13. Fact table types: transaction, periodic snapshot, accumulating
- **Transaction fact**: one row per event, written once and never changed. `fact_sales` (one row per order line). Most common, most flexible.
- **Periodic snapshot**: one row per thing per period, taken on a schedule, even if nothing happened. `fact_stock_month_end` above. Good for balances, inventory, headcount.
- **Accumulating snapshot**: one row per process instance, UPDATED as it passes milestones (ordered, shipped, delivered). Good for pipelines and lead times. Next section.
- Tip: pick the type from the question. "What happened?" -> transaction. "What was the state at month end?" -> periodic. "How long does each step take?" -> accumulating.

## 14. Accumulating snapshot: one row per order, updated as it progresses
- Grain: **one row per order**. It has several date keys (one per milestone). A milestone that has not happened yet points at `date_key = 0` ("not yet").
- The order-level `shipping_fee` lives here, at its true grain (this fixes the double count from section 04).
- Lags (`days_to_ship`, `days_to_deliver`) are NULL until the milestone happens. `julianday` differences give whole days (see 01-Basics/05-Dates-And-Times).

In [19]:
%%sql
CREATE TABLE fact_order_fulfilment (
    order_number INTEGER PRIMARY KEY,
    customer_key INTEGER NOT NULL REFERENCES dim_customer,
    order_date_key INTEGER NOT NULL REFERENCES dim_date,
    ship_date_key INTEGER NOT NULL REFERENCES dim_date,
    delivered_date_key INTEGER NOT NULL REFERENCES dim_date,
    shipping_fee INTEGER NOT NULL,
    days_to_ship INTEGER, days_to_deliver INTEGER
);
INSERT INTO fact_order_fulfilment
SELECT o.order_id, COALESCE(dc.customer_key, -1),
       CAST(strftime('%Y%m%d', o.order_date) AS INTEGER),
       COALESCE(CAST(strftime('%Y%m%d', o.ship_date) AS INTEGER), 0),
       COALESCE(CAST(strftime('%Y%m%d', o.delivered_date) AS INTEGER), 0),
       o.shipping_fee,
       CAST(julianday(o.ship_date) - julianday(o.order_date) AS INTEGER),
       CAST(julianday(o.delivered_date) - julianday(o.order_date) AS INTEGER)
FROM oltp_orders o
LEFT JOIN dim_customer dc ON dc.customer_id = o.customer_id
     AND o.order_date >= dc.valid_from AND o.order_date < dc.valid_to;
SELECT order_number, order_date_key, ship_date_key, delivered_date_key, shipping_fee, days_to_ship, days_to_deliver
FROM fact_order_fulfilment ORDER BY order_number;

order_number | order_date_key | ship_date_key | delivered_date_key | shipping_fee | days_to_ship | days_to_deliver
-------------+----------------+---------------+--------------------+--------------+--------------+----------------
1001         | 20250105       | 20250106      | 20250109           | 40           | 1            | 4              
1002         | 20250118       | 20250119      | 20250124           | 0            | 1            | 6              
1003         | 20250129       | 20250130      | 20250202           | 50           | 1            | 4              
1004         | 20250210       | 20250211      | 20250213           | 40           | 1            | 3              
1005         | 20250221       | 20250225      | 0                  | 60           | 4            | NULL           
1006         | 20250304       | 0             | 0                  | 40           | NULL         | NULL           
1007         | 20250312       | 20250313      | 20250317           | 0          

- Now a milestone arrives: order 1005 is delivered on 2025-03-02, order 1006 ships on 2025-03-06. We update the SAME rows. This is why it is "accumulating": rows are revisited.

In [20]:
%%sql
UPDATE oltp_orders SET delivered_date = '2025-03-02' WHERE order_id = 1005;
UPDATE oltp_orders SET ship_date = '2025-03-06' WHERE order_id = 1006;
UPDATE fact_order_fulfilment
SET ship_date_key = (SELECT COALESCE(CAST(strftime('%Y%m%d', o.ship_date) AS INTEGER), 0) FROM oltp_orders o WHERE o.order_id = order_number),
    delivered_date_key = (SELECT COALESCE(CAST(strftime('%Y%m%d', o.delivered_date) AS INTEGER), 0) FROM oltp_orders o WHERE o.order_id = order_number),
    days_to_ship = (SELECT CAST(julianday(o.ship_date) - julianday(o.order_date) AS INTEGER) FROM oltp_orders o WHERE o.order_id = order_number),
    days_to_deliver = (SELECT CAST(julianday(o.delivered_date) - julianday(o.order_date) AS INTEGER) FROM oltp_orders o WHERE o.order_id = order_number)
WHERE order_number IN (1005, 1006);
SELECT order_number, ship_date_key, delivered_date_key, days_to_ship, days_to_deliver
FROM fact_order_fulfilment WHERE order_number IN (1005, 1006) ORDER BY order_number;

order_number | ship_date_key | delivered_date_key | days_to_ship | days_to_deliver
-------------+---------------+--------------------+--------------+----------------
1005         | 20250225      | 20250302           | 4            | 9              
1006         | 20250306      | 0                  | 2            | NULL           
(2 rows)



- Lead-time report. `AVG` ignores NULL lags, so orders not yet delivered do not distort it. Orders still open are the ones with `delivered_date_key = 0`.

In [21]:
%%sql
SELECT d.year_month AS order_month, count(*) AS orders,
       ROUND(AVG(f.days_to_ship), 1) AS avg_days_to_ship,
       ROUND(AVG(f.days_to_deliver), 1) AS avg_days_to_deliver,
       SUM(f.delivered_date_key = 0) AS still_open,
       SUM(f.shipping_fee) AS shipping_fees
FROM fact_order_fulfilment f JOIN dim_date d ON d.date_key = f.order_date_key
GROUP BY d.year_month ORDER BY d.year_month;

order_month | orders | avg_days_to_ship | avg_days_to_deliver | still_open | shipping_fees
------------+--------+------------------+---------------------+------------+--------------
2025-01     | 3      | 1.0              | 4.7                 | 0          | 90           
2025-02     | 2      | 2.5              | 6.0                 | 0          | 100          
2025-03     | 3      | 1.3              | 4.0                 | 1          | 80           
(3 rows)



## 15. SCD type 1: overwrite
- Slowly changing dimensions (SCD) answer: "an attribute changed in the source, what do we do in the warehouse?"
- **Type 1** = overwrite the old value. Simple, no history. Right for corrections (typos) and for attributes nobody needs history of.
- Effect: every past fact now shows the new value, as if it had always been that way.

In [22]:
%%sql
UPDATE oltp_products SET name = 'Coffee Mug' WHERE product_id = 5;
UPDATE dim_product
SET product_name = (SELECT p.name FROM oltp_products p WHERE p.product_id = dim_product.product_id)
WHERE product_id IS NOT NULL
  AND product_name IS NOT (SELECT p.name FROM oltp_products p WHERE p.product_id = dim_product.product_id);
SELECT product_key, sku, product_name, category FROM dim_product ORDER BY product_key;

product_key | sku   | product_name | category  
------------+-------+--------------+-----------
-1          | n/a   | Unknown      | Unknown   
1           | P-100 | Notebook     | Stationery
2           | P-200 | Pen          | Stationery
3           | P-300 | Backpack     | Bags      
4           | P-400 | Desk Lamp    | Home      
5           | P-500 | Coffee Mug   | Home      
(6 rows)



## 16. SCD type 3: keep the previous value in a column
- **Type 3** = add a `previous_...` column. You keep exactly one step of history, in the same row.
- Good when the business asks "show me sales under the OLD and the NEW category structure" during a one-time re-org. Bad for repeated changes (older values are lost).
- Demo on a copy of the product dimension: Backpack moves from `Bags` to `Travel`.

In [23]:
%%sql
CREATE TABLE dim_product_t3 (
    product_key INTEGER PRIMARY KEY, product_name TEXT NOT NULL,
    category TEXT NOT NULL, previous_category TEXT
);
INSERT INTO dim_product_t3 (product_key, product_name, category)
SELECT product_key, product_name, category FROM dim_product WHERE product_key > 0;
-- the change: shift current into previous, then overwrite current
UPDATE dim_product_t3 SET previous_category = category, category = 'Travel' WHERE product_name = 'Backpack';
SELECT p.category AS new_category, p.previous_category AS old_category, SUM(f.gross_amount) AS revenue
FROM fact_sales f JOIN dim_product_t3 p USING (product_key)
GROUP BY p.category, p.previous_category ORDER BY p.category, p.previous_category;

new_category | old_category | revenue
-------------+--------------+--------
Home         | NULL         | 3150   
Stationery   | NULL         | 880    
Travel       | Bags         | 3160   
(3 rows)



- Notice the NULL `old_category` rows: products that never changed have no previous value. A second change to Backpack would overwrite `previous_category` and lose `Bags` for good.

In [24]:
%%sql
DROP TABLE dim_product_t3;

## 17. SCD type 2: add a new row (full history)
- **Type 2** = never overwrite. Close the old row (`valid_to`, `is_current = 0`) and insert a new row with a NEW surrogate key. Each fact later points at the version that was true when it happened.
- Two statements: (1) close current rows whose source attributes changed, (2) insert a new current row for every customer that has no current row. Run both in one transaction. For the history-table version inside an OLTP database see 06-Data-Modification/04-History-Audit-And-Temporal-Data.
- Scenario: Asha (customer 1) moves from Pune to Mumbai, effective 2025-02-15.
- `IS NOT` is used for the comparison so NULL-to-value changes are detected too (see 01-Basics/02-Filtering-And-Operators).

In [25]:
%%sql
UPDATE oltp_customers SET city = 'Mumbai' WHERE customer_id = 1;

BEGIN;
-- (1) close the current version of any customer whose tracked attributes changed
UPDATE dim_customer SET valid_to = '2025-02-15', is_current = 0
WHERE is_current = 1 AND customer_id IN (
    SELECT c.customer_id FROM oltp_customers c
    JOIN dim_customer d ON d.customer_id = c.customer_id AND d.is_current = 1
    WHERE c.city IS NOT d.city OR c.segment IS NOT d.segment OR c.name IS NOT d.customer_name);
-- (2) insert a new current version where none exists (same statement we used for the first load)
INSERT INTO dim_customer (customer_id, customer_name, city, segment, valid_from, valid_to, is_current)
SELECT c.customer_id, c.name, c.city, c.segment,
       CASE WHEN EXISTS (SELECT 1 FROM dim_customer d WHERE d.customer_id = c.customer_id)
            THEN '2025-02-15' ELSE '1900-01-01' END,
       '9999-12-31', 1
FROM oltp_customers c
WHERE NOT EXISTS (SELECT 1 FROM dim_customer d WHERE d.customer_id = c.customer_id AND d.is_current = 1)
ORDER BY c.customer_id;
COMMIT;
SELECT customer_key, customer_id, customer_name, city, valid_from, valid_to, is_current
FROM dim_customer WHERE customer_id = 1 ORDER BY valid_from;

customer_key | customer_id | customer_name | city   | valid_from | valid_to   | is_current
-------------+-------------+---------------+--------+------------+------------+-----------
1            | 1           | Asha          | Pune   | 1900-01-01 | 2025-02-15 | 0         
5            | 1           | Asha          | Mumbai | 2025-02-15 | 9999-12-31 | 1         
(2 rows)



- Asha now has two dimension rows with different surrogate keys. This is the whole point of surrogate keys: the business key `customer_id = 1` repeats, the warehouse key does not.
- The unique partial index `ux_dim_customer_current` would have rejected a second current row, protecting us from a half-done load.

## 18. SCD 2 and the facts: pick the version valid on the event date
- The facts loaded in section 10 all point at Asha's old row (the only one then). Orders after 2025-02-15 should point at the new row.
- Re-run the same fact load (the range join in the `LEFT JOIN dim_customer` does the work). Same statement as before, we only run it again.
- Then ask: revenue by city. Asha's January and early-February orders count for Pune, her March order for Mumbai. A type-1 overwrite would have moved all of it to Mumbai.

In [26]:
%%sql
DELETE FROM fact_sales;
INSERT INTO fact_sales
SELECT o.order_id,
       ROW_NUMBER() OVER (PARTITION BY l.order_id ORDER BY l.line_id),
       dd.date_key, COALESCE(dc.customer_key, -1), COALESCE(dp.product_key, -1), COALESCE(pr.profile_key, -1),
       l.qty, l.qty * l.unit_price, l.qty * p.unit_cost
FROM oltp_order_lines l
JOIN oltp_orders   o  ON o.order_id = l.order_id
JOIN oltp_products p  ON p.product_id = l.product_id
JOIN dim_date      dd ON dd.full_date = o.order_date
LEFT JOIN dim_customer      dc ON dc.customer_id = o.customer_id
                              AND o.order_date >= dc.valid_from AND o.order_date < dc.valid_to
LEFT JOIN dim_product       dp ON dp.product_id = l.product_id
LEFT JOIN dim_order_profile pr ON pr.channel = o.channel AND pr.payment_method = o.payment_method;

SELECT c.customer_name, c.city, c.valid_from, c.valid_to, SUM(f.gross_amount) AS revenue, count(DISTINCT f.order_number) AS orders
FROM fact_sales f JOIN dim_customer c USING (customer_key)
GROUP BY c.customer_key ORDER BY c.customer_name, c.valid_from;

customer_name | city   | valid_from | valid_to   | revenue | orders
--------------+--------+------------+------------+---------+-------
Asha          | Pune   | 1900-01-01 | 2025-02-15 | 1000    | 2     
Asha          | Mumbai | 2025-02-15 | 9999-12-31 | 500     | 1     
Ben           | Delhi  | 1900-01-01 | 9999-12-31 | 1230    | 2     
Chen          | Mumbai | 1900-01-01 | 9999-12-31 | 2900    | 2     
Dina          | Pune   | 1900-01-01 | 9999-12-31 | 1560    | 1     
(5 rows)



- Counting customers: `count(*)` on the dimension now counts versions. Use `COUNT(DISTINCT customer_id)` (the business key) for "how many customers".
- The accumulating fact also stores `customer_key`; refresh it the same way if you need the right version there.

In [27]:
%%sql
SELECT count(*) AS dimension_rows, count(DISTINCT customer_id) AS customers, SUM(is_current) AS current_rows
FROM dim_customer WHERE customer_key > 0;

dimension_rows | customers | current_rows
---------------+-----------+-------------
5              | 4         | 4           
(1 row)



## 19. Bridge tables for many-to-many
- Sometimes a fact relates to MANY dimension rows: a product has several tags (school, office, travel, gift). A foreign key cannot hold a list.
- Add a **bridge table** `bridge_product_tag` between `dim_product` and `dim_tag`, with a **weighting factor** that sums to 1.0 per product.
- Group by tag with the weight applied and the totals still tie out. Without weights, every product is counted once per tag, so the grand total is too high. (Pairs with the many-to-many pattern in 09-Design/01-Normalization-And-Design.)

In [28]:
%%sql
CREATE TABLE dim_tag (tag_key INTEGER PRIMARY KEY, tag TEXT NOT NULL UNIQUE);
CREATE TABLE bridge_product_tag (
    product_key INTEGER NOT NULL REFERENCES dim_product, tag_key INTEGER NOT NULL REFERENCES dim_tag,
    weight REAL NOT NULL CHECK (weight > 0 AND weight <= 1), PRIMARY KEY (product_key, tag_key)
) WITHOUT ROWID;
INSERT INTO dim_tag (tag) VALUES ('gift'),('office'),('school'),('travel');
INSERT INTO bridge_product_tag
WITH b(sku, tag, w) AS (VALUES
    ('P-100','school',0.5),('P-100','office',0.5),('P-200','school',0.5),('P-200','office',0.5),
    ('P-300','school',0.5),('P-300','travel',0.5),('P-400','office',1.0),
    ('P-500','office',0.5),('P-500','gift',0.5))
SELECT p.product_key, t.tag_key, b.w FROM b JOIN dim_product p ON p.sku = b.sku JOIN dim_tag t ON t.tag = b.tag;
-- a product's weights must add to 1: check it (no rows = good)
SELECT product_key, SUM(weight) AS total_weight FROM bridge_product_tag GROUP BY product_key HAVING ROUND(SUM(weight), 6) <> 1.0;

product_key | total_weight
------------+-------------
(0 rows)



In [29]:
%%sql
SELECT t.tag, ROUND(SUM(f.gross_amount * b.weight), 1) AS weighted_revenue, SUM(f.gross_amount) AS unweighted_revenue
FROM fact_sales f
JOIN bridge_product_tag b ON b.product_key = f.product_key
JOIN dim_tag t ON t.tag_key = b.tag_key
GROUP BY t.tag ORDER BY t.tag;
SELECT (SELECT SUM(gross_amount) FROM fact_sales) AS true_total,
       (SELECT ROUND(SUM(f.gross_amount * b.weight), 1) FROM fact_sales f JOIN bridge_product_tag b USING (product_key)) AS weighted_total,
       (SELECT SUM(f.gross_amount) FROM fact_sales f JOIN bridge_product_tag b USING (product_key)) AS unweighted_total;

tag    | weighted_revenue | unweighted_revenue
-------+------------------+-------------------
gift   | 675.0            | 1350              
office | 2915.0           | 4030              
school | 2020.0           | 4040              
travel | 1580.0           | 3160              
(4 rows)

true_total | weighted_total | unweighted_total
-----------+----------------+-----------------
7190       | 7190.0         | 12580           
(1 row)



## 20. Late-arriving data and incremental loads
- **Late-arriving dimension**: the fact shows up before its dimension row. Do not drop it and do not leave NULL: load it with the Unknown key (`-1`), then fix it when the dimension arrives.
- **Late-arriving fact**: an old event arrives now. With SCD 2 the range join still picks the version valid on the event date, so history stays right.
- **Incremental load**: instead of a full rebuild, insert only rows not yet in the fact (`NOT EXISTS` on the key, or a watermark like `order_id > max loaded`).
- Scenario: order 1009 arrives for a brand-new customer 5 whose dimension row has not been loaded yet.

In [30]:
%%sql
INSERT INTO oltp_customers VALUES (5, 'Esha', 'Delhi', 'Retail');
INSERT INTO oltp_orders VALUES (1009, 5, '2025-03-28', NULL, NULL, 'web', 'upi', 40);
INSERT INTO oltp_order_lines (order_id, product_id, qty, unit_price) VALUES (1009, 2, 10, 10), (1009, 5, 1, 150);

-- incremental: only lines of orders not yet in the fact
INSERT INTO fact_sales
SELECT o.order_id, ROW_NUMBER() OVER (PARTITION BY l.order_id ORDER BY l.line_id),
       dd.date_key, COALESCE(dc.customer_key, -1), COALESCE(dp.product_key, -1), COALESCE(pr.profile_key, -1),
       l.qty, l.qty * l.unit_price, l.qty * p.unit_cost
FROM oltp_order_lines l
JOIN oltp_orders   o  ON o.order_id = l.order_id
JOIN oltp_products p  ON p.product_id = l.product_id
JOIN dim_date      dd ON dd.full_date = o.order_date
LEFT JOIN dim_customer      dc ON dc.customer_id = o.customer_id AND o.order_date >= dc.valid_from AND o.order_date < dc.valid_to
LEFT JOIN dim_product       dp ON dp.product_id = l.product_id
LEFT JOIN dim_order_profile pr ON pr.channel = o.channel AND pr.payment_method = o.payment_method
WHERE NOT EXISTS (SELECT 1 FROM fact_sales f WHERE f.order_number = o.order_id);

SELECT f.order_number, f.customer_key, c.customer_name, f.gross_amount
FROM fact_sales f JOIN dim_customer c USING (customer_key) WHERE f.order_number = 1009 ORDER BY f.line_number;

order_number | customer_key | customer_name | gross_amount
-------------+--------------+---------------+-------------
1009         | -1           | Unknown       | 100         
1009         | -1           | Unknown       | 150         
(2 rows)



- The fact is safe (not lost, not NULL) but labelled Unknown. Now the customer dimension catches up (same SCD-2 insert as before), and we repair the Unknown rows by looking up the right key.

In [31]:
%%sql
INSERT INTO dim_customer (customer_id, customer_name, city, segment, valid_from, valid_to, is_current)
SELECT c.customer_id, c.name, c.city, c.segment,
       CASE WHEN EXISTS (SELECT 1 FROM dim_customer d WHERE d.customer_id = c.customer_id)
            THEN '2025-02-15' ELSE '1900-01-01' END, '9999-12-31', 1
FROM oltp_customers c
WHERE NOT EXISTS (SELECT 1 FROM dim_customer d WHERE d.customer_id = c.customer_id AND d.is_current = 1)
ORDER BY c.customer_id;

UPDATE fact_sales
SET customer_key = (
    SELECT dc.customer_key FROM oltp_orders o
    JOIN dim_customer dc ON dc.customer_id = o.customer_id AND o.order_date >= dc.valid_from AND o.order_date < dc.valid_to
    WHERE o.order_id = fact_sales.order_number)
WHERE customer_key = -1;

SELECT f.order_number, c.customer_name, c.city, f.gross_amount
FROM fact_sales f JOIN dim_customer c USING (customer_key) WHERE f.order_number = 1009 ORDER BY f.line_number;
SELECT count(*) AS unknown_customer_rows_left FROM fact_sales WHERE customer_key = -1;

order_number | customer_name | city  | gross_amount
-------------+---------------+-------+-------------
1009         | Esha          | Delhi | 100         
1009         | Esha          | Delhi | 150         
(2 rows)

unknown_customer_rows_left
--------------------------
0                         
(1 row)



- Good practice: a monitoring query that counts rows pointing at Unknown members, run after every load. A growing number means a broken feed.

## 21. Conformed dimensions and drilling across
- A **conformed dimension** is one shared dimension (same keys, same meaning) used by several facts. `dim_date` and `dim_product` are used by sales AND stock AND returns. That is what lets you compare them.
- Add `fact_returns` (grain: one row per returned order line). It reuses `dim_date`, `dim_product`, `dim_customer`.
- **Drill across**: to compare two facts, aggregate EACH fact to the shared dimension first, then join the small results. Joining the two fact tables directly multiplies rows.

In [32]:
%%sql
CREATE TABLE fact_returns (
    order_number INTEGER NOT NULL, line_number INTEGER NOT NULL,
    date_key INTEGER NOT NULL REFERENCES dim_date, product_key INTEGER NOT NULL REFERENCES dim_product,
    customer_key INTEGER NOT NULL REFERENCES dim_customer,
    quantity INTEGER NOT NULL, refund_amount INTEGER NOT NULL, PRIMARY KEY (order_number, line_number)
) WITHOUT ROWID;
INSERT INTO fact_returns
WITH r(ord, ln, d, q, refund) AS (VALUES (1003,2,'2025-02-05',1,150),(1004,2,'2025-02-20',1,150),(1007,1,'2025-03-20',1,600),(1001,2,'2025-01-15',2,20))
SELECT f.order_number, f.line_number, dd.date_key, f.product_key, f.customer_key, r.q, r.refund
FROM r JOIN fact_sales f ON f.order_number = r.ord AND f.line_number = r.ln
JOIN dim_date dd ON dd.full_date = r.d;
-- WRONG: joining two facts directly. Each sales row is repeated per matching return row.
SELECT p.product_name, SUM(s.gross_amount) AS wrong_revenue, SUM(r.refund_amount) AS refunds
FROM fact_sales s JOIN fact_returns r ON r.product_key = s.product_key
JOIN dim_product p ON p.product_key = s.product_key
GROUP BY p.product_name ORDER BY p.product_name;

product_name | wrong_revenue | refunds
-------------+---------------+--------
Coffee Mug   | 3000          | 1200   
Desk Lamp    | 1800          | 1200   
Pen          | 430           | 80     
(3 rows)



- Right: drill across. Each fact is summed per product in its own subquery, then the two small results are joined on the conformed key. Products with no returns need a `LEFT JOIN`.

In [33]:
%%sql
WITH sales AS (SELECT product_key, SUM(gross_amount) AS revenue FROM fact_sales GROUP BY product_key),
     rets  AS (SELECT product_key, SUM(refund_amount) AS refunds FROM fact_returns GROUP BY product_key)
SELECT p.product_name, s.revenue, COALESCE(r.refunds, 0) AS refunds,
       ROUND(100.0 * COALESCE(r.refunds, 0) / s.revenue, 1) AS refund_pct
FROM sales s JOIN dim_product p USING (product_key) LEFT JOIN rets r USING (product_key)
ORDER BY p.product_name;

product_name | revenue | refunds | refund_pct
-------------+---------+---------+-----------
Backpack     | 3160    | 0       | 0.0       
Coffee Mug   | 1500    | 300     | 20.0      
Desk Lamp    | 1800    | 600     | 33.3      
Notebook     | 550     | 0       | 0.0       
Pen          | 430     | 20      | 4.7       
(5 rows)



## 22. Naming conventions for a data mart
- Prefix by role so anyone can tell tables apart at a glance: `dim_` dimensions, `fact_` facts, `bridge_` bridges, `stg_` staging (raw landing copies), `oltp_` or the source schema name for sources.
- Keys: `<entity>_key` = surrogate (warehouse-made integer), `<entity>_id` = business key from the source, `date_key` = `YYYYMMDD` integer.
- Fact names say the grain: `fact_sales` (line), `fact_order_fulfilment` (order), `fact_stock_month_end` (product x month-end). Put the grain in a comment or a table description too.
- Measures carry their meaning and unit in the name: `gross_amount`, `quantity`, `days_to_ship`. Flags are 0/1 integers: `is_weekend`, `is_current`.
- Use the same column names for the same thing in every table (`product_key` everywhere). That is what makes `USING (product_key)` and conformed dimensions work.
- Inspect what we built by listing tables from `sqlite_master`:

In [34]:
%%sql
SELECT name, type FROM sqlite_master
WHERE type = 'table' AND (name LIKE 'dim\_%' ESCAPE '\' OR name LIKE 'fact\_%' ESCAPE '\' OR name LIKE 'bridge\_%' ESCAPE '\')
ORDER BY name;

name                  | type 
----------------------+------
bridge_product_tag    | table
dim_customer          | table
dim_date              | table
dim_order_profile     | table
dim_product           | table
dim_tag               | table
fact_order_fulfilment | table
fact_returns          | table
fact_sales            | table
fact_stock_month_end  | table
(10 rows)



## 23. When a star schema is overkill
- Small data, one source, few reports: a couple of views over the normalized tables are enough. A star adds an ETL job to build and keep correct.
- Heavy ad-hoc exploration of raw events (clickstream, logs): a single wide "one big table" or just the event table is often better than a star.
- The source is already analytics-shaped (one flat CSV): index it, do not model it.
- Real-time needs: a nightly star is stale. Query the OLTP replica or maintain a summary table.
- A star pays off when: many people ask many different questions, history must be kept stable (SCD), several sources must be combined, or the OLTP database must be protected from heavy reports.
- Lightweight middle ground: a view that flattens the joins. Analysts get one wide table, the source stays normalized. Compare with section 02: same kind of answer, short query. March is higher than before because the view also sees order 1009, added in section 20.

In [35]:
%%sql
CREATE VIEW v_sales_flat AS
SELECT o.order_date, substr(o.order_date, 1, 7) AS year_month, c.name AS customer, c.city,
       p.name AS product, p.category, l.qty, l.qty * l.unit_price AS revenue
FROM oltp_order_lines l JOIN oltp_orders o USING (order_id)
JOIN oltp_customers c USING (customer_id) JOIN oltp_products p USING (product_id);
SELECT year_month, category, SUM(revenue) AS revenue FROM v_sales_flat GROUP BY 1, 2 ORDER BY 1, 2;

year_month | category   | revenue
-----------+------------+--------
2025-01    | Bags       | 800    
2025-01    | Home       | 900    
2025-01    | Stationery | 300    
2025-02    | Bags       | 1560   
2025-02    | Home       | 600    
2025-02    | Stationery | 100    
2025-03    | Bags       | 800    
2025-03    | Home       | 1800   
2025-03    | Stationery | 580    
(9 rows)



- Caveat: a view always shows the CURRENT customer city (no history, like SCD type 1) and recomputes the joins on each query. Both limits are what a real star fixes.

## 24. Design checklist
- 1. Pick the business process (sales, shipments, stock). One fact per process.
- 2. Declare the grain in one sentence. Write it down.
- 3. Choose the dimensions that are true at that grain. Add Unknown members.
- 4. Choose the measures that are true at that grain. Make them additive where you can, store parts not ratios.
- 5. For each dimension attribute decide: overwrite (type 1), history (type 2), or previous value (type 3).
- 6. Build a date dimension. Use integer date keys.
- 7. Load with `INSERT ... SELECT`, reconcile totals against the source, monitor Unknown-key counts.
- 8. Many-to-many: bridge with weights. Several facts: conformed dimensions, drill across.
- Continue with 09-Design/03-Schema-Design-Case-Studies for three full OLTP designs, and 07-Performance/01-Indexes-And-Query-Plans for indexing the fact table.